# Friction damped example

In [ ]:
import numpy as np
from scipy import linalg as la
from scipy import optimize as op
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size': 12,'font.family':'serif'})

In [ ]:
%matplotlib notebook

## Parameter choices

In [ ]:
m = 1
k = 1
kc = 10
muN = 1

## Construct M, C, K without friction

In [ ]:
M = m*np.eye(2) # Mass matrix
C = 1e-2*M      # light Rayleigh damping for linear structure

K_no_friction = k*np.eye(2) # Stiffness matrix without friction
Kc = np.array([[1,-1],[-1,1]]) # Stiffness matrix structure for friction coupling

## Friction functions

In [ ]:
NT = 100 # Number of samples per period for friction force calculation
t = np.arange(0,2*np.pi,2*np.pi/NT)

# Calculate time-domain friction force for one period of excitation force
def friction_force(X):
    x = X*np.sin(t)
    xd = X*np.cos(t)
    f =np.zeros(NT)
    b = kc*X-muN
    b = np.max([b,0])
    
    f[xd>0] = kc*x[xd>0] + b
    f[xd<0] = kc*x[xd<0] - b
    f[f<-muN] = -muN
    f[f>muN] = muN
    
    return x,f

# Calculate linearised stiffness (numerical method)
def calc_K(X):
    x,f = friction_force(X)
    
    Xw = np.fft.rfft(x)
    Fw = np.fft.rfft(f)
    
    X1 = Xw[1]
    F1 = Fw[1]
    
    K = F1/X1
    
    return X1,F1,K
    
# Calculate error for a given value of normalised response 's'. Uses 'log(s)' for numerical stability.
def residue(log_s,w0,F0):
    
    s = np.exp(log_s)
    
    # Use numerical method for K:
    #X = muN/kc/s
    #X1,F1,K = calc_K(X) 
    
    # Use analytic method for K:
    if s < 1:
        Kreal = kc/np.pi * (np.arccos(1-2*s)-2*(1-2*s)*np.sqrt(s*(1-s)))
        Kimag = 4*kc/np.pi*s*(1-s)
        K = Kreal + 1j*Kimag
    else:
        K = kc
        
    K_with_friction = K_no_friction + Kc * K
    F = np.array([[F0],[0]])
    x = np.matmul(np.linalg.inv(K_with_friction +1j*w0*C - w0**2*M),F)
    X = np.abs(x[1]-x[0])
    s_test = muN / kc / X

    difference = np.log(s/s_test)
    
    return difference
    
# Calculate the response for a given input force amplitude over a given frequency range
def calc_response(w_range,F0):
    X = np.zeros(N)
    s = np.zeros(N)
    xx = np.zeros(N)
    x0 = 0.9
    for n,w in enumerate(w_range):
        x = op.fsolve(residue,x0,args=(w,F0))
        x0 = np.copy(x)
        #xx[n] = x
        s[n] = np.exp(x)
        X[n] = muN/kc/s[n]
        
    return X

## Eigenvalue analysis for a given response amplitude

In [ ]:
N = 1000
s_all = np.logspace(-2,0,N)
Kreal = np.zeros(N)
Kimag = np.zeros(N)
wn = np.zeros(N,dtype='complex')
for n,s in enumerate(s_all):
    
    # Use numerical method for K:
    #X = muN/kc/s
    #X1,F1,K = calc_K(X) 
    
    # Use analytic method for K:
    if s < 1:
        Kreal = kc/np.pi * (np.arccos(1-2*s)-2*(1-2*s)*np.sqrt(s*(1-s)))
        Kimag = 4*kc/np.pi*s*(1-s)
        K = Kreal + 1j*Kimag
    else:
        K = kc

    K_with_friction = K_no_friction + Kc * K

    wn2,Phi = la.eig(K_with_friction,M)  # eigh tells scipy that the matrices are symmetric
    
    i_min = np.argmin(wn2)
    i_max = np.argmax(wn2)
    wn[n] = np.sqrt(wn2[i_max])       # sqrt to get w from w**2

In [ ]:
eta = np.imag(wn)/np.real(wn)

In [ ]:
fig,ax = plt.subplots(figsize=(9,6))
ax.plot(np.real(wn),eta)
ax.set_xlabel('$\omega_n$')
ax.set_ylabel('$\eta_{eff}$')
ax.set_xlim([0,10])

## Compute 'frequency response' for a range of input force amplitudes

In [ ]:
fig,ax = plt.subplots(figsize=(9,6))
N = 1000

F0_range = np.linspace(0.01,3,10)
w_range = np.linspace(10,0.1,N)

for F0 in F0_range:
    X = calc_response(w_range,F0)
    ax.plot(w_range,20*np.log10(X/F0),label=f'$F_0=${F0:.2f}')

ax.set_xlabel('Driving frequency $\omega_0$')
ax.set_ylabel('Normalised response amplitude: $X/F_0$ (dB)')
ax.set_xlim([0,10])
ax.legend()